## Assignment 3: Fine-Tuning a Language Model for Text Classification

### Exercise 1: Introduction to Fine-Tuning a BERT Model for Text Classification
In this exercise, you will gain practical experience in fine-tuning a pre-trained language model for a text classification task. You will use the `bert-base-cased` model and the `yelp_review_full` dataset, which contains Yelp business reviews classified into five-star ratings. You are also free to choose a different pre-trained model or dataset if you prefer, but in that case, you will need to adapt the provided code accordingly.

Your goal is to fine-tune the model to predict the rating based on the text of the review. You are required to complete the following tasks:
- Load and prepare a text dataset for fine-tuning.
- Fine-tune a pre-trained language model for text classification.
- Evaluate the model's performance.
- Reflect on the model's strengths and weaknesses.

Example code snippets are provided below to help you complete the tasks. You can use them as a starting point for your solution. Alternatively, you are welcome to use other libraries and resources if you prefer.

### Exercise 2: Fine-Tuning a BERT Model for Text Classification with Your Own Dataset
In this exercise, you will use your own classified State of the Union dataset to fine-tune a BERT model for text classification. You can use the `bert-base-cased` model or select a different pre-trained model if you prefer. The goal is to fine-tune the model to predict the sentiment you assigned to paragraphs (or other subsets of the speeches in the first assignment) based on the text of the speech.

Although we do not provide specific code for this exercise, you should be able to reuse much of the code from Exercise 1 with some modifications. Here are some suggestions to guide your approach:
- **Dataset Creation**: Create a dataset with text samples and corresponding sentiment labels. This can be a binary classification task (e.g., positive vs. negative sentiment) or a multi-class classification task (e.g., positive vs. neutral vs. negative sentiment). You will need to choose appropriate cutoffs for the sentiment scores to create the labels.
- **Fine-Tuning**: Fine-tune a pre-trained language model for text classification. You can adapt the code from the previous exercise to work with your dataset and classification task.
- **Evaluation**: Evaluate the model's performance using the same evaluation metrics from Exercise 1 or other suitable metrics. How well does this approach work for your dataset compared to the simple approach you used in the first assignment? If you did not use a simple approach in the first assignment, you should do one now to compare to the results here. **Make sure you are using the same data for the different approaches.** Use e.g., a dictionary based approach, a boolean approach, or a simple machine learning model like the Vader sentiment analysis model. 
- **Analysis**: Discuss the strengths and weaknesses of your model. How well does it perform on the sentiment classification task? What factors influence its performance? Do you see any benefits or limitations of using a pre-trained language model for this task over a simpler approach?


**Note 1**: *This exercise, especially Exercise 1, can be computationally demanding. During the lecture "Practical Day 2," we will discuss how to potentially run this on a server to make it feasible to include more training data.*

**Note 2**: *The code provided below is based on the Hugging Face documentation. You can find more details here: [Hugging Face Transformers Training Documentation](https://huggingface.co/docs/transformers/en/training).*

In [1]:
# Import the load_dataset function from the datasets library to load and work with text datasets (you may need to install the library first)
from datasets import load_dataset

# Load the 'yelp_review_full' dataset, which contains Yelp business reviews with five-star ratings
dataset = load_dataset("yelp_review_full")

# Access and display the 100th sample from the 'train' split of the dataset
dataset["train"][0:3]


ModuleNotFoundError: No module named 'datasets'

In [ ]:
import pandas as pd
from datasets import Dataset

dataset = pd.read_csv('data.csv')

dataset = pd.DataFrame([['test', 0],
                        ['test2', 1]], columns=['text', 'label'])

# Convert the pandas DataFrame to a Hugging Face Dataset
dataset_hf = Dataset.from_pandas(dataset)

In [ ]:
dataset_hf

Dataset({
    features: ['text', 'label'],
    num_rows: 2
})

In [ ]:
# Import the AutoTokenizer class from the transformers library for loading a tokenizer (you need to install the tokenizer library first)
from transformers import AutoTokenizer

# Load a pre-trained tokenizer from the 'google-bert/bert-base-cased' model
# This tokenizer will be used to tokenize the text data for input into the model
# We will use this model for fine-tuning on the Yelp review dataset, if you have a different model in mind, you can replace it here
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

# Define a function to tokenize input examples from the dataset
def tokenize_function(examples):
    # Tokenize the 'text' field of the examples with padding and truncation
    # 'padding="max_length"' pads the input sequences to the model's maximum length
    # 'truncation=True' ensures that input sequences longer than the maximum length are truncated
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply the tokenize_function to the entire dataset using the map() method
# 'batched=True' processes examples in batches for faster computation
tokenized_datasets = dataset_hf.map(tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

## Train with PyTorch 

In [ ]:
# Import PyTorch (needs to be installed)
import torch

In [ ]:
# Remove the text column because the model does not accept raw text as an input
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Rename the label column to labels because the model expects the argument to be named labels
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set the format of the dataset to return PyTorch tensors instead of lists
tokenized_datasets.set_format("torch")

In [ ]:
tokenized_datasets

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2
})

In [ ]:
# Create a smaller subset of the dataset to speed up the fine-tuning 
# You can increase the size of the subset or use the full dataset for better results but it will take longer
# Here, we are using 1000 examples each for training and evaluation
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

KeyError: "Column train not in the dataset. Current columns in the dataset: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']"

In [ ]:
# Import the DataLoader class from PyTorch's torch.utils.data module
# DataLoader allows efficient batching, shuffling, and loading of data for model training and evaluation
from torch.utils.data import DataLoader

# Create a DataLoader for the training dataset
# small_train_dataset is assumed to be a subset of your tokenized training data
# shuffle=True shuffles the data at each epoch to improve model generalization
# batch_size=8 specifies that each batch will contain 8 samples
train_dataloader = DataLoader(small_train_dataset, shuffle=True, batch_size=8)

# Create a DataLoader for the evaluation dataset
# small_eval_dataset is assumed to be a subset of your tokenized evaluation data
# shuffle is not used for evaluation data to maintain consistency in data order
# batch_size=8 specifies that each batch will contain 8 samples
eval_dataloader = DataLoader(small_eval_dataset, batch_size=8)

In [ ]:
# Import the AutoModelForSequenceClassification class from the transformers library
# This class allows loading a pre-trained transformer model specifically for classification tasks
from transformers import AutoModelForSequenceClassification

# Load a pre-trained BERT model ('google-bert/bert-base-cased') for sequence classification
# The num_labels parameter specifies the number of output labels/classes for classification
# Here, num_labels=5 indicates that the model is being fine-tuned for a task with five classes (e.g., a five-class classification problem)
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Import the AdamW optimizer from PyTorch's torch.optim module
# AdamW is a variant of the Adam optimizer with weight decay, commonly used for fine-tuning transformers
from torch.optim import AdamW

# Create an optimizer for fine-tuning the model
# optimizer updates the model parameters during training to minimize the loss function
# model.parameters() specifies which parameters (weights and biases) to optimize
# lr=5e-5 sets the learning rate for the optimizer, controlling the step size at each update
optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
# Import the get_scheduler function from the transformers library
# This function allows the creation of different learning rate schedulers
from transformers import get_scheduler

# Define the number of epochs (full passes through the training data)
num_epochs = 3

# Calculate the total number of training steps (batches)
# This is the product of the number of epochs and the number of batches in the training DataLoader
num_training_steps = num_epochs * len(train_dataloader)

# Create a learning rate scheduler
# This scheduler linearly decreases the learning rate from its initial value (set in the optimizer) to zero over the course of training
# name="linear" specifies a linear learning rate decay schedule
# optimizer specifies the optimizer to update (AdamW in this case)
# num_warmup_steps=0 means there is no warmup period where the learning rate increases from a small value to the initial value
# num_training_steps specifies the total number of steps for the decay schedule
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [ ]:
# Check if a GPU (CUDA) is available and set the device accordingly
# torch.device("cuda") selects a GPU if available for faster computation
# If a GPU is not available, it defaults to using the CPU for computation
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Move the model's parameters to the specified device (GPU or CPU)
# This ensures that all model computations happen on the chosen device
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Training loop

In [ ]:
# Note: on my machine, the following code block took about 30 minutes to run

# Import tqdm for creating a progress bar to visually track training progress
# tqdm.auto automatically chooses the appropriate version (e.g., Jupyter-friendly) for display
from tqdm.auto import tqdm

# Create a progress bar for the total number of training steps
# This will visually track the training progress over all epochs and batches
progress_bar = tqdm(range(num_training_steps))

# Set the model to training mode
# This activates behaviors like dropout and batch normalization in training mode
model.train()

# Loop over the number of epochs (full passes through the dataset)
for epoch in range(num_epochs):
    # Loop over each batch in the training DataLoader
    for batch in train_dataloader:
        # Move each batch of data to the specified device (CPU or GPU)
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass: Pass the input data through the model to obtain outputs
        outputs = model(**batch)

        # Compute the loss (automatically computed based on the specified loss function for the model)
        loss = outputs.loss

        # Backpropagation: Compute gradients for model parameters with respect to the loss
        loss.backward()

        # Update model parameters using the optimizer based on computed gradients
        optimizer.step()

        # Update the learning rate using the learning rate scheduler
        lr_scheduler.step()

        # Reset the gradients of the model parameters to zero
        # This prevents the accumulation of gradients from multiple backward passes
        optimizer.zero_grad()

        # Update the progress bar to reflect the completion of another training step
        progress_bar.update(1)


  0%|          | 0/375 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Import the evaluate library to facilitate the computation of metrics
import evaluate

# Load the accuracy metric for evaluation
# This metric will be used to evaluate the model's performance on the evaluation dataset
metric = evaluate.load("accuracy")

# Set the model to evaluation mode
# This deactivates training-specific behaviors such as dropout for more consistent predictions
model.eval()

# Loop over each batch in the evaluation DataLoader
for batch in eval_dataloader:
    # Move the batch of data to the specified device (CPU or GPU) for computation
    batch = {k: v.to(device) for k, v in batch.items()}

    # Disable gradient calculation during evaluation to save memory and computation
    # This ensures no gradients are computed, as they are not needed during evaluation
    with torch.no_grad():
        # Perform a forward pass through the model to obtain outputs (predictions)
        outputs = model(**batch)

    # Extract logits (raw output scores) from the model's output
    logits = outputs.logits

    # Convert the logits to predicted class labels
    # torch.argmax selects the index of the maximum value along the specified dimension (dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    # Add the batch of predictions and reference (true) labels to the metric for computation
    metric.add_batch(predictions=predictions, references=batch["labels"])

# Compute and return the final accuracy metric based on all evaluated batches
metric.compute()


## Make a prediction using a new text example

In [ ]:
# Function to make predictions for input text
def predict_class(input_text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        input_text,
        return_tensors="pt",        # Return PyTorch tensors
        padding="max_length",       # Pad to the maximum length used during training
        truncation=True,            # Truncate if the input exceeds the maximum length
        max_length=128              # Ensure this matches the max length used during training
    )
    
    # Move input tensors to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Put the model in evaluation mode
    model.eval()
    
    # Make predictions without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits (raw scores) from the model's output
    logits = outputs.logits
    
    # Convert logits to predicted class (taking the index of the max logit)
    predicted_class = torch.argmax(logits, dim=-1).item()
    
    return predicted_class

# Example usage
input_text = "The food at this restaurant was absolutely amazing!"
predicted_class = predict_class(input_text, model, tokenizer, device)

# Display the predicted class, 0 to 4, based on the input text 4 being the highest rating
print(f"The predicted class for the input text is: {predicted_class}")
